#### Contextual Compression Retriever


In [17]:
import os

from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

load_dotenv()
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    raise ValueError("Set HF_TOKEN or HUGGINGFACEHUB_API_TOKEN in .env")

In [18]:
docs = [
    Document(
        page_content="""
The Grand Canyon is one of the most visited natural wonders in the world.
Photosynthesis is the process by which green plants convert sunlight into energy.
Millions of tourists travel to see it every year. The rocks date back millions of years.
""",
        metadata={"source": "Doc1"},
    ),

    Document(
        page_content="""
In medieval Europe, castles were built primarily for defense.
The chlorophyll in plant cells captures sunlight during photosynthesis.
Knights wore armor made of metal. Siege weapons were often used to breach castle walls.
""",
        metadata={"source": "Doc2"},
    ),

    Document(
        page_content="""
Basketball was invented by Dr. James Naismith in the late 19th century.
It was originally played with a soccer ball and peach baskets. NBA is now a global league.
""",
        metadata={"source": "Doc3"},
    ),

    Document(
        page_content="""
The history of cinema began in the late 1800s. Silent films were the earliest form.
Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
""",
        metadata={"source": "Doc4"},
    ),
]

In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

endpoint = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    provider="featherless-ai",
    huggingfacehub_api_token=hf_token,
    max_new_tokens=256,
    temperature=0,
    do_sample=False,
)

# Featherless supports this model through chat completion, not text generation.
llm = ChatHuggingFace(llm=endpoint)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10857.06it/s]


In [20]:
vector_store = FAISS.from_documents(docs, embeddings)


##### Normal base retriever

In [21]:
base_retriever = vector_store.as_retriever(
    search_kwargs = {"k":4}
)

COMPRESSOR VIA LLM

In [22]:
#Compressor
compressor = LLMChainExtractor.from_llm(llm)

COMPRESSOR RETRIEVER

In [23]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [24]:
query = "What is photosynthesis?"
compressed_result = compression_retriever.invoke(query)

In [25]:
for i, doc in enumerate(compressed_result):
    print(f"\n ---Result{i+1} ---")
    print(doc.page_content)


 ---Result1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

 ---Result2 ---
> The chlorophyll in plant cells captures sunlight during photosynthesis.

 ---Result3 ---
Photosynthesis does not occur in animal cells.
